# 02 - Analisis exploratorio (Ejercicio 4)

Las preguntas (4.1) y su justificacion estan en `docs/eda.md`; aqui se ejecutan las consultas (4.2) de `sql/02_eda/`, se grafican y se interpretan (4.4).
La vista `trips` unifica yellow y green; **no se eliminan registros**: cada consulta aplica y declara sus filtros de calidad.

In [ ]:
import os, sys
from pathlib import Path
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60, "display.width", 180, "display.max_colwidth", 60)

# Dentro del contenedor el proyecto esta en /workspace; fuera, en la carpeta padre de notebooks/.
RAIZ = Path("/workspace") if Path("/workspace/sql").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / "scripts"))
import taxi_common as tc
DATOS = tc.data_dir().as_posix()          # carpeta data/ segun donde se ejecute
from document_queries import abrir

# FUENTE = "parquet": la vista `trips` lee los Parquet de data/raw (normalizados por taxi_common.py).
# FUENTE = "tabla"  : la vista `trips` apunta a la tabla de data/processed/taxi.duckdb (abierta en SOLO LECTURA).
# Las MISMAS consultas sirven para ambas fuentes. Para cambiar: LAB_FUENTE=tabla antes de iniciar Jupyter.
FUENTE = os.environ.get("LAB_FUENTE", "parquet")
con = abrir(FUENTE, str(tc.db_por_defecto()))

def sql(archivo: str) -> pd.DataFrame:
    """Ejecuta un archivo de sql/ y devuelve un DataFrame. La consulta vive en el .sql, no en el notebook."""
    texto = (RAIZ / "sql" / archivo).read_text(encoding="utf-8").replace("/workspace/data", DATOS)
    return con.execute(texto).df()

print("DuckDB", duckdb.__version__, "| fuente:", FUENTE, "| archivos:", len(tc.listar_archivos()))

## P1 - ¿Como evoluciona la demanda mes a mes?

Tendencia y estacionalidad (comportamiento temporal).

Consulta: `sql/02_eda/d10_tendencia_mensual.sql`

In [ ]:
m = sql("02_eda/d10_tendencia_mensual.sql")
ax = m.pivot(index="mes", columns="taxi_type", values="viajes").plot(figsize=(10, 4), marker="o", ylabel="viajes", title="Viajes por mes")
plt.show()
m.head()

**Interpretacion / decision:** <!-- COMPLETAR con lo que muestre el resultado real -->

## P2 - ¿Cuando se concentra la demanda?

Dia de la semana (0 = domingo) y hora.

Consulta: `sql/02_eda/d05_patron_hora_dia.sql`

In [ ]:
h = sql("02_eda/d05_patron_hora_dia.sql")
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, (tipo, g) in zip(axes, h.groupby("taxi_type")):
    m = g.pivot(index="dia_semana", columns="hora", values="viajes")
    im = ax.imshow(m.values, aspect="auto", cmap="viridis")
    ax.set_title(f"{tipo}: viajes por dia y hora"); ax.set_xlabel("hora"); ax.set_ylabel("dia (0=dom)")
    fig.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

**Interpretacion / decision:** <!-- COMPLETAR con lo que muestre el resultado real -->

## P3 - ¿Como es un viaje tipico?

Estadisticos, distribucion de la distancia, duracion y velocidad (caracteristicas de los viajes y distribucion de valores).

Consulta: `sql/02_eda/d01_estadisticos_numericos.sql`

In [ ]:
sql("02_eda/d01_estadisticos_numericos.sql")

In [ ]:
d = sql("02_eda/d03_distribucion_distancia.sql")
d.pivot(index="rango_millas", columns="taxi_type", values="porcentaje").plot.bar(figsize=(8, 4), ylabel="% de viajes", title="Distribucion de la distancia")
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

In [ ]:
sql("02_eda/d04_duracion_y_velocidad.sql")

**Interpretacion / decision:** <!-- COMPLETAR -->

## P4 - ¿En que se diferencian yellow y green?

Comparacion directa en una tabla (volumen, distancia, duracion, tarifa, propina, tarjeta, aeropuertos).

Consulta: `sql/02_eda/d11_comparacion_yellow_green.sql`

In [ ]:
sql("02_eda/d11_comparacion_yellow_green.sql")

**Interpretacion / decision:** <!-- COMPLETAR con lo que muestre el resultado real -->

## P5 - ¿Como pagan y cuanto dejan de propina?

Variables relacionadas con el pago. Ojo: la propina en efectivo no se registra.

Consulta: `sql/02_eda/d06_pago_y_propinas.sql`

In [ ]:
sql("02_eda/d06_pago_y_propinas.sql")

**Interpretacion / decision:** <!-- COMPLETAR con lo que muestre el resultado real -->

## P6 - ¿Que zonas concentran recogidas y destinos?

Zonas por ID de la TLC.

Consulta: `sql/02_eda/d07_zonas_recogida_destino.sql`

In [ ]:
sql("02_eda/d07_zonas_recogida_destino.sql")

**Interpretacion / decision:** <!-- COMPLETAR con lo que muestre el resultado real -->

## P7 - ¿Que variables se relacionan?

Correlaciones entre distancia, tarifa, total, propina y pasajeros.

Consulta: `sql/02_eda/d08_correlaciones.sql`

In [ ]:
sql("02_eda/d08_correlaciones.sql")

**Interpretacion / decision:** <!-- COMPLETAR con lo que muestre el resultado real -->

## P8 - ¿Cuantos registros son inconsistentes o atipicos?

Conteo de registros sospechosos y atipicos de tarifa por IQR.

Consulta: `sql/02_eda/d02_calidad_datos.sql`

In [ ]:
sql("02_eda/d02_calidad_datos.sql")

In [ ]:
sql("02_eda/d09_atipicos_iqr.sql")

**Interpretacion / decision:** <!-- COMPLETAR: que filtros se aplican y por que -->

## Hallazgos relevantes (4.5, minimo tres)

<!-- COMPLETAR con cifras reales, citando la consulta. Por ejemplo:
1. ...
2. ...
3. ... -->